##Install Dependencies


In [1]:
!pip install -q ragas langchain langchain-openai langchain-huggingface psycopg2-binary pgvector langchain-postgres datasets nest_asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.0/213.0 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 106.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.4/252.4 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1

##The VertexAI Import Patch

In [2]:
import sys
import types

class DummyVertexAI: pass
class DummyChatVertexAI: pass

dummy_llms = types.ModuleType("langchain_community.llms")
dummy_llms.VertexAI = DummyVertexAI
sys.modules["langchain_community.llms"] = dummy_llms

dummy_chat_models = types.ModuleType("langchain_community.chat_models")
dummy_chat_models.ChatVertexAI = DummyChatVertexAI
sys.modules["langchain_community.chat_models"] = dummy_chat_models

dummy_chat_vertexai = types.ModuleType("langchain_community.chat_models.vertexai")
dummy_chat_vertexai.ChatVertexAI = DummyChatVertexAI
sys.modules["langchain_community.chat_models.vertexai"] = dummy_chat_vertexai

dummy_llms_vertexai = types.ModuleType("langchain_community.llms.vertexai")
dummy_llms_vertexai.VertexAI = DummyVertexAI
sys.modules["langchain_community.llms.vertexai"] = dummy_llms_vertexai

##Setup, Dual-Retrieval

In [4]:
import os
import json
import pandas as pd
import time
import nest_asyncio
from google.colab import userdata, drive
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_postgres import PGVector
from langchain_core.prompts import PromptTemplate

nest_asyncio.apply()
drive.mount('/content/drive')

# 1. SETUP PATHS & CREDENTIALS
DRIVE_PATH = '/content/drive/MyDrive/'
DATASET_PATH = DRIVE_PATH + 'AWMF_Golden_Dataset_200Q_Final.csv'
df = pd.read_csv(DATASET_PATH)

NEON_CONNECTION_STRING = userdata.get('NEON_DATABASE_URL')
os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')

# 2. INITIALIZE VECTOR STORE RETRIEVER
print("Connecting to PGVector Database...")
bge_embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3", model_kwargs={'device': 'cpu'})

vector_store = PGVector(
    embeddings=bge_embeddings,
    collection_name="awmf_baseline_bge",
    connection=NEON_CONNECTION_STRING,
    use_jsonb=True
)
# Setting k=10 for each model's independent sweep
retriever = vector_store.as_retriever(search_kwargs={"k": 10})

# 3. INITIALIZE THE 4-MODEL COHORT
print("Loading Specialist Models...")
claude_stage1  = ChatOpenAI(model="anthropic/claude-sonnet-4.6", api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1", temperature=0, max_tokens=500)
mistral_stage1 = ChatOpenAI(model="mistralai/mistral-large", api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1", temperature=0, max_tokens=500)
gemini_stage2  = ChatOpenAI(model="google/gemini-2.5-flash", api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1", temperature=0, max_tokens=2000)
gpt_stage3     = ChatOpenAI(model="openai/gpt-4o-mini", api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1", temperature=0,max_tokens=1500)

# 4. ARCHITECTURAL PROMPTS
translation_expansion_prompt = PromptTemplate(
    template="""You are an expert medical search term generator.
First, translate the following English medical question into German.
Then, generate 3 to 4 highly formal German clinical synonyms, related medical conditions, or official MeSH terms (Query Expansion) that would likely appear in a formal clinical guideline.
Output ONLY the translated question and the synonyms combined as a single continuous search string. Do not include bullet points, labels, or introductory text.

English Question:
{question}""",
    input_variables=["question"]
)

stage2_extraction_prompt = PromptTemplate(
    template="""You are an expert medical data extractor. Read the provided German clinical guidelines and extract ALL clinical facts, criteria, or medical recommendations necessary to answer the English medical question.
Your response must be written in English.
Be highly technical, literal, and strict. Do NOT assume, extrapolate, or add external clinical knowledge. If the text does not contain the answer, state 'Information not found'.

Context (German):
{context}

Question (English):
{question}

Extracted English Facts:""",
    input_variables=["context", "question"]
)

stage3_polish_prompt = PromptTemplate(
    template="""You are a medical AI communicator translating raw extraction data into a clean clinical answer.
Take the following raw extracted medical facts and format them into a highly accurate, professional, and readable medical summary answering the user's question.
Maintain complete fidelity to the facts provided. Do not add external recommendations.

Raw Extracted Facts:
{raw_facts}

Original Question:
{question}

Final Answer (English):""",
    input_variables=["raw_facts", "question"]
)

print("Parallel-Fusion Compound system initialized successfully!")

Mounted at /content/drive
Connecting to PGVector Database...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Loading Specialist Models...
Parallel-Fusion Compound system initialized successfully!


##The Parallel-Fusion Execution Loop

In [5]:
output_file = f"{DRIVE_PATH}PARALLEL_FUSION_SYSTEM_results.json"
results = {"question": [], "answer": [], "contexts": [], "ground_truth": []}

print("RUNNING DUAL-QUERY FUSION EXPERIMENT (200 QUESTIONS)...")

for index, row in df.iterrows():
    english_question = row['English_Open_Question']
    english_ground_truth = row['English_Correct_Text']

    try:
        # ---- STAGE 1: PARALLEL QUERY EXPANSION ----
        formatted_prompt = translation_expansion_prompt.format(question=english_question)

        # Both models process query variants simultaneously
        claude_query = claude_stage1.invoke(formatted_prompt).content.strip()
        mistral_query = mistral_stage1.invoke(formatted_prompt).content.strip()

        # ---- DUAL DATABASE RETRIEVAL ----
        docs_claude = retriever.invoke(claude_query)
        docs_mistral = retriever.invoke(mistral_query)

        # ---- QUERY FUSION & DE-DUPLICATION ----
        seen_chunks = set()
        fused_contexts = []

        # Interleave hits to preserve vector ranking order from both perspectives
        for doc_c, doc_m in zip(docs_claude, docs_mistral):
            if doc_c.page_content not in seen_chunks:
                seen_chunks.add(doc_c.page_content)
                fused_contexts.append(doc_c.page_content)
            if doc_m.page_content not in seen_chunks:
                seen_chunks.add(doc_m.page_content)
                fused_contexts.append(doc_m.page_content)

        context_string = "\n\n".join(fused_contexts)

        # ---- STAGE 2: GEMINI STRICT EXTRACTION ----
        f_stage2 = stage2_extraction_prompt.format(context=context_string, question=english_question)
        raw_facts = gemini_stage2.invoke(f_stage2).content.strip()

        # ---- STAGE 3: GPT-4o-MINI COMMUNICATOR ----
        f_stage3 = stage3_polish_prompt.format(raw_facts=raw_facts, question=english_question)
        final_polished_answer = gpt_stage3.invoke(f_stage3).content.strip()

        # ---- PERSIST ----
        results["question"].append(english_question)
        results["answer"].append(final_polished_answer)
        results["contexts"].append(fused_contexts)
        results["ground_truth"].append(english_ground_truth)

        with open(output_file, 'w') as f:
            json.dump(results, f)

        if (index + 1) % 20 == 0:
            print(f"Progress: {index + 1}/{len(df)}")
            print(f" -> Claude Search Variant: {claude_query[:60]}...")
            print(f" -> Mistral Search Variant: {mistral_query[:60]}...")
            print(f" -> Unique Fused Chunks Retained: {len(fused_contexts)}")
            print(f" -> Gemini Output Length: {len(raw_facts)} chars")
            print(f" -> GPT Polished Answer: {final_polished_answer[:90]}...\n")

        time.sleep(2)

    except Exception as e:
        print(f"Error at index {index}: {e}")
        time.sleep(5)
        continue

print("Parallel-Fusion Pipeline Execution Complete!")

RUNNING DUAL-QUERY FUSION EXPERIMENT (200 QUESTIONS)...
Progress: 20/200
 -> Claude Search Variant: Ein 48-jähriger Mann wird wegen plötzlich einsetzender Atemn...
 -> Mistral Search Variant: „48-jähriger Mann wird mit seit 6 Stunden bestehender akuter...
 -> Unique Fused Chunks Retained: 16
 -> Gemini Output Length: 4398 chars
 -> GPT Polished Answer: In this clinical scenario, the patient presents with acute difficulty breathing, orthopnea...

Progress: 40/200
 -> Claude Search Variant: Ein 59-jähriger Mann wird wegen progressiver Gelenkschmerzen...
 -> Mistral Search Variant: „59-jähriger Patient mit progredienten Gelenkbeschwerden: Sc...
 -> Unique Fused Chunks Retained: 17
 -> Gemini Output Length: 4210 chars
 -> GPT Polished Answer: The clinical presentation of the 59-year-old man, characterized by progressive joint pain,...

Progress: 60/200
 -> Claude Search Variant: Ein 7-jähriger Junge wird zur Nachsorgeuntersuchung beim Kin...
 -> Mistral Search Variant: „Ein 7-jähriger Jung

##Ragas Evaluation

In [6]:
from datasets import Dataset, Features, Value, Sequence
from ragas import evaluate
from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig

print("GRADING THE PARALLEL-FUSION PIPELINE MEASUREMENTS...")

with open(output_file, 'r') as f:
    data = json.load(f)

evaluation_features = Features({
    "question": Value("string"),
    "answer": Value("string"),
    "contexts": Sequence(Value("string")),
    "ground_truth": Value("string"),
})

eval_dataset = Dataset.from_dict(data, features=evaluation_features)

judge_llm = LangchainLLMWrapper(gpt_stage3)
ragas_embeddings = LangchainEmbeddingsWrapper(bge_embeddings)

eval_results = evaluate(
    dataset=eval_dataset,
    metrics=[context_precision, context_recall, faithfulness, answer_relevancy],
    llm=judge_llm,
    embeddings=ragas_embeddings,
    run_config=RunConfig(timeout=300, max_workers=2, max_retries=5)
)

res_df = eval_results.to_pandas()
res_df.to_csv(f"{DRIVE_PATH}PARALLEL_FUSION_FINAL_EVALUATION.csv", index=False)

print("\n FUSION PIPELINE BENCHMARK COMPLETE!")
print(res_df[['context_precision', 'context_recall', 'faithfulness', 'answer_relevancy']].mean().round(3))

/tmp/ipykernel_1261/2525447180.py:3: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
/tmp/ipykernel_1261/2525447180.py:3: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
/tmp/ipykernel_1261/2525447180.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import context_precis

GRADING THE PARALLEL-FUSION PIPELINE MEASUREMENTS...


Evaluating:   0%|          | 0/800 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[90]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)
ERROR:ragas.executor:Exception raised in Job[198]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)
ERROR:ragas.executor:Exception raised in Job[462]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)
ERROR:ragas.executor:Exception raised in Job[470]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)
ERROR:ragas.executor:Exception raised in Job[602]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)
ERROR:ragas.executor:Exception raised in Job[614]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)
ERROR:ragas.executor:Exception rais


 FUSION PIPELINE BENCHMARK COMPLETE!
context_precision    0.181
context_recall       0.268
faithfulness         0.227
answer_relevancy     0.567
dtype: float64
